# Ropedia Academy — D4 · Semantic & open-vocabulary mapping

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChaoYue0307/ropedia-academy/blob/main/notebooks/D4.ipynb)

> **Bayesian multi-frame label fusion (storing a distribution that corrects 2D flicker) plus open-vocabulary querying by matching language to per-element CLIP features.**
>
> 贝叶斯多帧标签融合（存储能纠正 2D 闪烁的分布），再加把语言与逐元素 CLIP 特征匹配的开放词汇查询。

This is the lesson's core example — **self-contained and runnable end to end**. It builds toy tensors, performs the lesson's key computation, and prints a real result, so you learn the concept by executing it.

Colab's default runtime already includes `torch`, `numpy`, and `networkx`, so just press **Run all** — every cell should go green. Sizes are shrunk to run on CPU; swap in a real batch and the same code scales up.

🔗 Full lesson (with the interactive demo & key terms): https://chaoyue0307.github.io/ropedia-academy/lesson/D4

In [ ]:
import torch, torch.nn.functional as F

# Fuse per-frame 2D labels into a 3D map element via BAYESIAN multi-frame fusion.
n_classes, true_class = 4, 2
torch.manual_seed(0)
logp = torch.zeros(n_classes)                        # uniform prior (log space)
for _ in range(15):                                  # 15 noisy 2D observations
    obs = F.one_hot(torch.tensor(true_class), n_classes).float()*2 + torch.randn(n_classes)
    logp = logp + F.log_softmax(obs, -1)             # accumulate evidence
    logp = logp - logp.logsumexp(0)                  # renormalize (keep a distribution)
print("fused class probs:", logp.exp().round(decimals=3).tolist(), "-> argmax", logp.argmax().item())
print("multi-frame voting corrects 2D flicker (store the DISTRIBUTION, not one label)")

# OPEN-VOCABULARY: attach a CLIP feature per element; query by language similarity.
voxel_clip = F.normalize(torch.randn(5, 16), dim=1)   # 5 mapped elements
query = F.normalize(torch.randn(16), dim=0)           # e.g. CLIP("something to sit on")
sims = voxel_clip @ query
print("best language-query match: element", sims.argmax().item(), "sim", round(sims.max().item(), 3))

### Where to go next

- Swap the toy tensors for a real batch and watch the shapes flow through.
- Open the matching lesson for the math and an interactive figure: https://chaoyue0307.github.io/ropedia-academy/lesson/D4
- Browse every notebook: https://github.com/ChaoYue0307/ropedia-academy/tree/main/notebooks